In [20]:
import os
import tempfile

import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.distributions.empirical_distribution import ECDF

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

In [21]:
def pd_read_parquet(artifact_path: str) -> pd.DataFrame:
    '''
    This function should read a Parquet file located at the provided path and return a pandas DataFrame.
    '''
    return pd.read_parquet(artifact_path)

path = 'data (1).parquet'
df = pd_read_parquet(path)
df.head()

,user_id,ts,active_time_m7,active_time_m14,active_time_s7,active_time_s14,activity_percent_m7,activity_percent_m14,activity_percent_s7,activity_percent_s14,...,phq_q4,phq_q5,phq_q6,phq_q7,phq_q8,phq_q9,phq2_total,phq8_total,phq9_total,deployment
0,us-east-1:1b087523-b47b-421c-97c4-74da80564598,2018-09-25 18:17:26.647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,0.0,1.0,1.0,1.0,NaN,3.0,9.0,NaN,hr-rct-cue-1
1,us-east-1:ea986d8e-e87d-458b-a798-1999fbbff05b,2018-09-26 17:13:52.394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,3.0,3.0,3.0,NaN,4.0,21.0,NaN,hr-rct-cue-1
2,us-east-1:8dfca9b6-22b6-4155-8a70-ce3e1211a684,2018-09-27 19:04:15.124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,3.0,3.0,1.0,NaN,6.0,22.0,NaN,hr-rct-cue-1
3,us-east-1:77f43bb4-fe8e-4a50-a4fc-41bc14b11f9d,2018-10-01 22:01:32.622,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,1.0,2.0,2.0,0.0,NaN,2.0,10.0,NaN,hr-rct-cue-1
4,us-east-1:4940dfa3-1c01-49f2-9cdd-029291d7b7a8,2018-10-05 21:08:52.441,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,3.0,3.0,1.0,0.0,NaN,6.0,18.0,NaN,hr-rct-cue-1


In [22]:
#print the shape of the dataset.
print(df.shape)
print('Count of rows:', df.shape[0])
print('Count of columns:', df.shape[1])

(6820, 158)
Count of rows: 6820
Count of columns: 158


In [23]:
# For stakeholders: Filter the dataset to include only the 'hr-ascent-1' deployment
df = df[df.deployment == 'hr-ascent-1']
df.head()

,user_id,ts,active_time_m7,active_time_m14,active_time_s7,active_time_s14,activity_percent_m7,activity_percent_m14,activity_percent_s7,activity_percent_s14,...,phq_q4,phq_q5,phq_q6,phq_q7,phq_q8,phq_q9,phq2_total,phq8_total,phq9_total,deployment
0,us-east-1:db841a21-c4fd-cf09-4379-5af0b6185e0e,2024-03-23 21:58:52.242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,2.0,3.0,3.0,1.0,6.0,23.0,24.0,hr-ascent-1
1,us-east-1:db841a21-c48b-cd1c-563f-eb23aa6c2768,2024-03-26 14:46:43.386,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,1.0,2.0,2.0,1.0,5.0,19.0,20.0,hr-ascent-1
2,us-east-1:db841a21-c46a-c6fd-ca13-1faabac2c46d,2024-03-27 16:19:40.526,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,1.0,0.0,4.0,12.0,12.0,hr-ascent-1
3,us-east-1:db841a21-c4b2-c6b9-91ed-370b2da2aa86,2024-03-27 16:42:15.485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.0,3.0,2.0,3.0,1.0,1.0,6.0,21.0,22.0,hr-ascent-1
4,us-east-1:db841a21-c438-c76a-5078-c47a75e6b0c3,2024-03-28 14:52:36.017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,0.0,2.0,7.0,7.0,hr-ascent-1


In [24]:
#print the shape of the dataset.
print(df.shape)
print('Count of rows:', df.shape[0])
print('Count of columns:', df.shape[1])

(2924, 158)
Count of rows: 2924
Count of columns: 158


### The PHQ-9 is a depression screening tool with nine questions:

1. Anhedonia (loss of interest)

2. Depressed mood

3. Sleep issues

4. Fatigue

5. Appetite changes

6. Guilt/worthlessness

7. Concentration issues

8. Psychomotor agitation/retardation

9. Suicidal thoughts

### The original mappings were:

- Sleep: PHQ-1, PHQ-3, PHQ-4

- Activity: PHQ-1, PHQ-4, PHQ-7

- Sociability: PHQ-1, PHQ-6, PHQ-9

- Device Usage: PHQ-2, PHQ-5, PHQ-8


### 1. Sleep Features
Key Insight: Sleep disruptions directly impact circadian rhythms and trickle down to activity/sociability.
    
    PHQ-3 ("Trouble sleeping") is the direct correlate for sleep disruptions.
    PHQ-4 ("Feeling tired/low energy") reflects the downstream effect of poor sleep.

    PHQ-9 Mapping: Primary: PHQ-3 (core sleep disturbance), Secondary: PHQ-4 (fatigue from poor sleep)


##### Features
    sleep_duration_m7, sleep_duration_m14, sleep_duration_s7, sleep_duration_s14
    sleep_start_m7, sleep_start_m14, sleep_start_s7, sleep_start_s14
    sleep_end_m7, sleep_end_m14, sleep_end_s7, sleep_end_s14
    sleep_interruptions_m7, sleep_interruptions_m14, sleep_interruptions_s7, sleep_interruptions_s14
    sleep_routine_index_m7, sleep_routine_index_m14, sleep_routine_index_s7, sleep_routine_index_s14
    chronotype_score_m7, chronotype_score_m14, chronotype_score_s7, chronotype_score_s14

### 2. Activity Features
Key Insight: Physical activity correlates with energy levels and mental engagement.

    PHQ-1 ("Little interest/pleasure") aligns with reduced physical engagement.
    PHQ-4 ("Low energy") correlates with activity levels.

    PHQ-9 Mapping: PHQ-1 (Loss of interest), PHQ-4 (Fatigue).

##### Features
    active_time_m7, active_time_m14, active_time_s7, active_time_s14
    activity_percent_m7, activity_percent_m14, activity_percent_s7, activity_percent_s14
    ec_Activity_m7, ec_Activity_m14, ec_Activity_s7, ec_Activity_s14
    ec_PedometerData_m7, ec_PedometerData_m14, ec_PedometerData_s7, ec_PedometerData_s14
    step_count_m7, step_count_m14, step_count_s7, step_count_s14
    steps_most_active_10h_m7, steps_most_active_10h_m14, steps_most_active_10h_s7, steps_most_active_10h_s14
    steps_least_active_5h_m7, steps_least_active_5h_m14, steps_least_active_5h_s7, steps_least_active_5h_s14
    steps_relative_amplitude_m7, steps_relative_amplitude_m14, steps_relative_amplitude_s7, steps_relative_amplitude_s14
    walking_rate_m7, walking_rate_m14, walking_rate_s7, walking_rate_s14
    total_activity_duration_m7, total_activity_duration_m14, total_activity_duration_s7, total_activity_duration_s14

### 3. Sociability Features
Key Insight: Movement outside home and location diversity reflect social engagement.

    PHQ-1 ("Withdrawal from social activities") strongly aligns with location entropy/clusters.
    PHQ-6 ("Feeling worthless") may correlate with isolation (time at home).


    PHQ-9 Mapping: PHQ-1 (social withdrawal), PHQ-6 (isolation → guilt).

##### Features
    time_at_home_m7, time_at_home_m14, time_at_home_s7, time_at_home_s14
    time_at_location_1_m7, time_at_location_1_m14, ..., time_at_location_5_s14
    num_location_clusters_m7, num_location_clusters_m14, num_location_clusters_s7, num_location_clusters_s14
    loc_entropy_m7, loc_entropy_m14, loc_entropy_s7, loc_entropy_s14
    travel_diameter_m7, travel_diameter_m14, travel_diameter_s7, travel_diameter_s14
    ec_Location_m7, ec_Location_m14, ec_Location_s7, ec_Location_s14
    total_location_duration_m7, total_location_duration_m14, total_location_duration_s7, total_location_duration_s14

### 4. Device Usage Features (New Bucket)
Key Insight: Screen time may reflect avoidance behaviors or reduced physical/social activity.
    PHQ-2 ("Feeling down") correlates with increased passive screen time.
    PHQ-8 ("Fidgeting/slowing") maps to screen unlocks (agitation) or low usage (retardation).

    PHQ-9 Mapping: PHQ-2 (Mood), PHQ-8 (Psychomotor agitation).

##### Features
    ec_DeviceDisplayStatus_m7, ec_DeviceDisplayStatus_m14, ec_DeviceDisplayStatus_s7, ec_DeviceDisplayStatus_s14
    display_events_m7, display_events_m14, display_events_s7, display_events_s14
    screen_unlocks_m7, screen_unlocks_m14, screen_unlocks_s7, screen_unlocks_s14
    total_display_duration_m7, total_display_duration_m14, total_display_duration_s7, total_display_duration_s14
    device_use_percent_m7, device_use_percent_m14, device_use_percent_s7, device_use_percent_s14

### 5. Demographic Features
Key Insight: Age, occupation, and ethnicity may moderate mental health outcomes.

##### Features
    age_bucket, age_num, sex, gender, housemates, occupation, school, phone_carry, full_time, ethnicity, ethnicity_white, age_binary, chronotype

### 6. Other/General Features
Purpose: Metadata or composite metrics.

##### Features
    user_id, ts, ec_total_m7, ec_total_m14, ec_total_s7, ec_total_s14, deployment

### Notes on Weighting for BVC
    Sleep likely has the highest weight due to its trickle-down effect on activity/sociability.
    Activity and Sociability weights can be derived from regression coefficients in the model.
    Device Usage may act as a confounder (e.g., high screen time → low activity).

In [46]:
def categorize_features():
    # Feature Bucket Definitions
    buckets = {
        "Sleep": [
            # Sleep duration/patterns
            'sleep_duration_m7', 'sleep_duration_m14', 'sleep_duration_s7', 'sleep_duration_s14',
            'sleep_start_m7', 'sleep_start_m14', 'sleep_start_s7', 'sleep_start_s14',
            'sleep_end_m7', 'sleep_end_m14', 'sleep_end_s7', 'sleep_end_s14',
            'sleep_interruptions_m7', 'sleep_interruptions_m14', 'sleep_interruptions_s7', 'sleep_interruptions_s14',
            
            # Sleep quality metrics
            'sleep_routine_index_m7', 'sleep_routine_index_m14', 'sleep_routine_index_s7', 'sleep_routine_index_s14',
            'chronotype_score_m7', 'chronotype_score_m14', 'chronotype_score_s7', 'chronotype_score_s14'
        ],
        
        "Activity": [
            # Physical activity metrics
            'active_time_m7', 'active_time_m14', 'active_time_s7', 'active_time_s14',
            'activity_percent_m7', 'activity_percent_m14', 'activity_percent_s7', 'activity_percent_s14',
            'ec_Activity_m7', 'ec_Activity_m14', 'ec_Activity_s7', 'ec_Activity_s14',
            'ec_PedometerData_m7', 'ec_PedometerData_m14', 'ec_PedometerData_s7', 'ec_PedometerData_s14',
            
            # Step-based metrics
            'step_count_m7', 'step_count_m14', 'step_count_s7', 'step_count_s14',
            'steps_most_active_10h_m7', 'steps_most_active_10h_m14', 'steps_most_active_10h_s7', 'steps_most_active_10h_s14',
            'steps_least_active_5h_m7', 'steps_least_active_5h_m14', 'steps_least_active_5h_s7', 'steps_least_active_5h_s14',
            'steps_relative_amplitude_m7', 'steps_relative_amplitude_m14', 'steps_relative_amplitude_s7', 'steps_relative_amplitude_s14',
            
            # Movement metrics
            'walking_rate_m7', 'walking_rate_m14', 'walking_rate_s7', 'walking_rate_s14',
            'total_activity_duration_m7', 'total_activity_duration_m14', 'total_activity_duration_s7', 'total_activity_duration_s14'
        ],
        
        "Sociability": [
            # Location patterns
            'time_at_home_m7', 'time_at_home_m14', 'time_at_home_s7', 'time_at_home_s14',
            
            # Location clusters (all variations)
            *[f'time_at_location_{i}_m7' for i in range(1,6)],
            *[f'time_at_location_{i}_m14' for i in range(1,6)],
            *[f'time_at_location_{i}_s7' for i in range(1,6)],
            *[f'time_at_location_{i}_s14' for i in range(1,6)],
            
            # Location diversity metrics
            'num_location_clusters_m7', 'num_location_clusters_m14', 'num_location_clusters_s7', 'num_location_clusters_s14',
            'loc_entropy_m7', 'loc_entropy_m14', 'loc_entropy_s7', 'loc_entropy_s14',
            'travel_diameter_m7', 'travel_diameter_m14', 'travel_diameter_s7', 'travel_diameter_s14',
            
            # Location event metrics
            'ec_Location_m7', 'ec_Location_m14', 'ec_Location_s7', 'ec_Location_s14',
            'total_location_duration_m7', 'total_location_duration_m14', 'total_location_duration_s7', 'total_location_duration_s14'
        ],
        
        "Device Usage": [
            # Screen interaction metrics
            'ec_DeviceDisplayStatus_m7', 'ec_DeviceDisplayStatus_m14', 'ec_DeviceDisplayStatus_s7', 'ec_DeviceDisplayStatus_s14',
            'display_events_m7', 'display_events_m14', 'display_events_s7', 'display_events_s14',
            'screen_unlocks_m7', 'screen_unlocks_m14', 'screen_unlocks_s7', 'screen_unlocks_s14',
            
            # Device usage patterns
            'total_display_duration_m7', 'total_display_duration_m14', 'total_display_duration_s7', 'total_display_duration_s14',
            'device_use_percent_m7', 'device_use_percent_m14', 'device_use_percent_s7', 'device_use_percent_s14'
        ],
        
        "Demographic": [
            # User profile data
            'age_bucket', 'age_num', 'sex', 'gender', 'housemates', 'occupation', 
            'school', 'phone_carry', 'full_time', 'ethnicity', 'ethnicity_white', 
            'age_binary', 'chronotype'
        ],
        
        "Other": [
            # Technical/metadata fields
            'user_id', 'ts', 'deployment',
            
            # Composite event counts
            'ec_total_m7', 'ec_total_m14', 'ec_total_s7', 'ec_total_s14'
        ],
        
        "Target": [
            # PHQ-9 questionnaire items
            'phq_q1', 'phq_q2', 'phq_q3', 'phq_q4', 'phq_q5', 
            'phq_q6', 'phq_q7', 'phq_q8', 'phq_q9',
            
            # PHQ scores and categories
            'phq2_total', 'phq8_total', 'phq9_total', 'phq_category'
        ]
    }

    # Validation check to ensure all features are categorized
    all_features = sum(buckets.values(), [])
    original_features = df.columns

    # Check for uncategorized features
    uncategorized = [f for f in original_features if f not in all_features]
    if uncategorized:
        print("Warning: Uncategorized features detected:", uncategorized)
        buckets["Uncategorized"] = uncategorized

    return buckets

# Usage Example:
feature_buckets = categorize_features()

# Print bucket sizes
for category, features in feature_buckets.items():
    print(f"{category}: {len(features)} features")

# To get features for a specific bucket:
sleep_features = feature_buckets["Sleep"]
activity_features = feature_buckets["Activity"]
target_features = feature_buckets["Target"]

Sleep: 24 features
Activity: 40 features
Sociability: 44 features
Device Usage: 20 features
Demographic: 13 features
Other: 7 features
Target: 13 features


In [47]:
feature_buckets

{'Sleep': ['sleep_duration_m7',
  'sleep_duration_m14',
  'sleep_duration_s7',
  'sleep_duration_s14',
  'sleep_start_m7',
  'sleep_start_m14',
  'sleep_start_s7',
  'sleep_start_s14',
  'sleep_end_m7',
  'sleep_end_m14',
  'sleep_end_s7',
  'sleep_end_s14',
  'sleep_interruptions_m7',
  'sleep_interruptions_m14',
  'sleep_interruptions_s7',
  'sleep_interruptions_s14',
  'sleep_routine_index_m7',
  'sleep_routine_index_m14',
  'sleep_routine_index_s7',
  'sleep_routine_index_s14',
  'chronotype_score_m7',
  'chronotype_score_m14',
  'chronotype_score_s7',
  'chronotype_score_s14'],
 'Activity': ['active_time_m7',
  'active_time_m14',
  'active_time_s7',
  'active_time_s14',
  'activity_percent_m7',
  'activity_percent_m14',
  'activity_percent_s7',
  'activity_percent_s14',
  'ec_Activity_m7',
  'ec_Activity_m14',
  'ec_Activity_s7',
  'ec_Activity_s14',
  'ec_PedometerData_m7',
  'ec_PedometerData_m14',
  'ec_PedometerData_s7',
  'ec_PedometerData_s14',
  'step_count_m7',
  'step_cou

In [43]:
sleep_features

['sleep_duration_m7',
 'sleep_duration_m14',
 'sleep_duration_s7',
 'sleep_duration_s14',
 'sleep_start_m7',
 'sleep_start_m14',
 'sleep_start_s7',
 'sleep_start_s14',
 'sleep_end_m7',
 'sleep_end_m14',
 'sleep_end_s7',
 'sleep_end_s14',
 'sleep_interruptions_m7',
 'sleep_interruptions_m14',
 'sleep_interruptions_s7',
 'sleep_interruptions_s14',
 'sleep_routine_index_m7',
 'sleep_routine_index_m14',
 'sleep_routine_index_s7',
 'sleep_routine_index_s14',
 'chronotype_score_m7',
 'chronotype_score_m14',
 'chronotype_score_s7',
 'chronotype_score_s14']

In [26]:
24+40+44+20
# +13+7+13

128

In [27]:
feature_buckets

{'Sleep': ['sleep_duration_m7',
  'sleep_duration_m14',
  'sleep_duration_s7',
  'sleep_duration_s14',
  'sleep_start_m7',
  'sleep_start_m14',
  'sleep_start_s7',
  'sleep_start_s14',
  'sleep_end_m7',
  'sleep_end_m14',
  'sleep_end_s7',
  'sleep_end_s14',
  'sleep_interruptions_m7',
  'sleep_interruptions_m14',
  'sleep_interruptions_s7',
  'sleep_interruptions_s14',
  'sleep_routine_index_m7',
  'sleep_routine_index_m14',
  'sleep_routine_index_s7',
  'sleep_routine_index_s14',
  'chronotype_score_m7',
  'chronotype_score_m14',
  'chronotype_score_s7',
  'chronotype_score_s14'],
 'Activity': ['active_time_m7',
  'active_time_m14',
  'active_time_s7',
  'active_time_s14',
  'activity_percent_m7',
  'activity_percent_m14',
  'activity_percent_s7',
  'activity_percent_s14',
  'ec_Activity_m7',
  'ec_Activity_m14',
  'ec_Activity_s7',
  'ec_Activity_s14',
  'ec_PedometerData_m7',
  'ec_PedometerData_m14',
  'ec_PedometerData_s7',
  'ec_PedometerData_s14',
  'step_count_m7',
  'step_cou

In [28]:


def scale_feature_buckets(df, feature_buckets):
    """
    Performs bucketed scaling while handling missing values and preserving non-scaled columns
    """
    # Create a copy of the dataframe to preserve original data
    scaled_df = df.copy()
    
    # Buckets to scale (excluding Demographic, Other, and Target)
    buckets_to_scale = ['Sleep', 'Activity', 'Sociability', 'Device Usage']
    
    # Create a combined list of all columns to exclude from scaling
    exclude_columns = sum([feature_buckets[b] for b in ['Demographic', 'Other', 'Target']], [])
    
    for bucket in buckets_to_scale:
        bucket_features = [f for f in feature_buckets[bucket] if f in scaled_df.columns]
        
        if not bucket_features:
            continue
            
        # Step 1: Handle missing values using mean imputation
        # imputer = SimpleImputer(strategy='mean')
        # scaled_df[bucket_features] = imputer.fit_transform(scaled_df[bucket_features])
        
        # Step 2: Perform Standard Scaling
        scaler = StandardScaler(with_mean=False, with_std=True)
        scaled_data = scaler.fit_transform(scaled_df[bucket_features])
        
        # Convert back to DataFrame to preserve column names
        scaled_df[bucket_features] = pd.DataFrame(scaled_data, 
                                                columns=bucket_features,
                                                index=scaled_df.index)
        
    # Preserve non-scaled columns (original values where not scaled)
    for col in exclude_columns:
        if col in scaled_df.columns:
            scaled_df[col] = df[col]  # Keep original values
    
    return scaled_df

# Usage Example:
feature_buckets = categorize_features()  # From previous implementation
scaled_df = scale_feature_buckets(df, feature_buckets)

In [29]:
import os
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def process_and_analyze_buckets(df, feature_buckets, target='phq8_total'):
    """Main function to process buckets and run analyses"""
    # Create age binary column
    def get_binary_age(age_bucket: str):
        if pd.isnull(age_bucket):
            return None
        if '85' in str(age_bucket):
            return 'old'
        try:
            ub = int(str(age_bucket).strip().split('-')[-1])
            return 'young' if ub < 45 else 'old'
        except:
            return None
    
    df['age_binary'] = df.age_bucket.apply(get_binary_age)
    
    # Define buckets to process
    buckets_to_process = ['Sleep', 'Activity', 'Sociability', 'Device Usage']
    
    # Create base output directory
    output_dir = os.path.join(os.getcwd(), "regression_results")
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each bucket
    for bucket in buckets_to_process:
        print(f"\nProcessing {bucket} bucket...")
        bucket_features = [f for f in feature_buckets[bucket] if f in df.columns]
        
        # Skip if no features found
        if not bucket_features:
            print(f"No features found for {bucket} bucket")
            continue
            
        # Create bucket-specific directory
        bucket_dir = os.path.join(output_dir, bucket.lower().replace(" ", "_"))
        os.makedirs(bucket_dir, exist_ok=True)
        
        # Process each feature in the bucket
        for feature in bucket_features:
            try:
                # Prepare temporary dataframe
                temp_df = df[['user_id', target, 'sex', 'age_bucket', 'age_binary'] + [feature]].copy()
                
                # Handle missing values and filtering
                temp_df = temp_df.dropna(subset=[feature, target, 'sex', 'age_binary'])
                temp_df = temp_df[temp_df.sex != 'Other']
                
                # Check if data remains
                if temp_df.empty:
                    print(f"Skipping {feature} - no data after cleaning")
                    continue
                    
                # Scale the feature
                scaler = StandardScaler(with_mean=False, with_std=True)
                # imputer = SimpleImputer(strategy='mean')
                
                # Impute and scale
                # temp_df[feature] = imputer.fit_transform(temp_df[[feature]])
                temp_df[feature] = scaler.fit_transform(temp_df[[feature]])
                
                # Ensure proper data types
                temp_df['sex'] = temp_df.sex.astype('category')
                temp_df['age_binary'] = temp_df.age_binary.astype('category')
                
                # Build model formula
                formula = f"{target} ~ Q('{feature}') : C(sex) : C(age_binary)"
                
                # Fit model
                model = smf.ols(formula, data=temp_df).fit()
                
                # Save results
                log_file = os.path.join(bucket_dir, f"{feature}.txt")
                with open(log_file, 'w') as f:
                    f.write(str(model.summary()))
                    
                print(f"Processed {feature} in {bucket}")
                
            except Exception as e:
                print(f"Failed processing {feature}: {str(e)}")
                continue

# Get feature buckets from previous implementation
feature_buckets = categorize_features()

# Run the analysis pipeline
process_and_analyze_buckets(
    df=df,
    feature_buckets=feature_buckets,
    target='phq8_total'  # Can change to other targets if needed
)


Processing Sleep bucket...
Processed sleep_duration_m7 in Sleep
Processed sleep_duration_m14 in Sleep
Processed sleep_duration_s7 in Sleep
Processed sleep_duration_s14 in Sleep
Processed sleep_start_m7 in Sleep
Processed sleep_start_m14 in Sleep
Processed sleep_start_s7 in Sleep
Processed sleep_start_s14 in Sleep
Processed sleep_end_m7 in Sleep
Processed sleep_end_m14 in Sleep
Processed sleep_end_s7 in Sleep
Processed sleep_end_s14 in Sleep
Processed sleep_interruptions_m7 in Sleep
Processed sleep_interruptions_m14 in Sleep
Processed sleep_interruptions_s7 in Sleep
Processed sleep_interruptions_s14 in Sleep
Processed sleep_routine_index_m7 in Sleep
Processed sleep_routine_index_m14 in Sleep
Processed sleep_routine_index_s7 in Sleep
Processed sleep_routine_index_s14 in Sleep
Processed chronotype_score_m7 in Sleep
Processed chronotype_score_m14 in Sleep
Processed chronotype_score_s7 in Sleep
Processed chronotype_score_s14 in Sleep

Processing Activity bucket...
Processed active_time_m7 

## OLS Regression results obtained here.

In [30]:
def parse_enhanced_ols_summary(file_path):
    """Extracts full metrics from OLS summary"""
    with open(file_path, 'r') as f:
        content = f.read()
    
    metrics = {
        'r_squared': None,
        'adj_r_squared': None,
        'features': []
    }
    
    # Extract model metrics
    r2_match = re.search(r'R-squared:\s+(\d+\.\d+)', content)
    adj_r2_match = re.search(r'Adj\. R-squared:\s+(\d+\.\d+)', content)
    
    if r2_match:
        metrics['r_squared'] = float(r2_match.group(1))
    if adj_r2_match:
        metrics['adj_r_squared'] = float(adj_r2_match.group(1))
    
    # Extract feature-level metrics
    table_match = re.search(r'=+\n(.*?)\n\n', content, re.DOTALL)
    if table_match:
        rows = table_match.group(1).split('\n')[2:]
        for row in rows:
            cols = re.split(r'\s{2,}', row.strip())
            if len(cols) < 6:
                continue
            
            try:
                feature = cols[0].replace("Q('", "").replace("')", "")
                metrics['features'].append({
                    'feature': feature,
                    'coefficient': float(cols[1]),
                    'p_value': float(cols[4])
                })
            except:
                continue
    
    return metrics

def generate_feature_rankings(results_dir='regression_results'):
    all_data = []
    
    for bucket in os.listdir(results_dir):
        bucket_path = os.path.join(results_dir, bucket)
        if not os.path.isdir(bucket_path):
            continue
        
        for file in os.listdir(bucket_path):
            if file.endswith('.txt'):
                file_path = os.path.join(bucket_path, file)
                metrics = parse_enhanced_ols_summary(file_path)
                
                for feat in metrics['features']:
                    all_data.append({
                        'feature': feat['feature'],
                        'coefficient': feat['coefficient'],
                        'p_value': feat['p_value'],
                        'r_squared': metrics['r_squared'],
                        'adj_r_squared': metrics['adj_r_squared'],
                        'bucket': bucket
                    })
    
    df = pd.DataFrame(all_data)
    
    # Filter and rank
    ranked_df = (
        df[df['p_value'] < 0.05]
        .assign(abs_coeff=lambda x: x['coefficient'].abs())
        .sort_values(['abs_coeff', 'p_value'], ascending=[False, True])
        .drop('abs_coeff', axis=1)
    )
    
    return ranked_df

# Generate and display results
feature_rankings = generate_feature_rankings()
feature_rankings.shape

(385, 6)

In [31]:
feature_rankings_filter = feature_rankings[ (feature_rankings.feature != 'Intercept') & (feature_rankings.r_squared >= 0.02) ].sort_values(by='r_squared', ascending=False)
feature_rankings_filter

,feature,coefficient,p_value,r_squared,adj_r_squared,bucket
353,sleep_end_s14:C(sex)[Female]:C(age_binary)[young],1.2404,0.000,0.068,0.061,sleep
351,sleep_end_s14:C(sex)[Female]:C(age_binary)[old],1.1913,0.001,0.068,0.061,sleep
331,sleep_duration_s14:C(sex)[Female]:C(age_binary...,0.9129,0.003,0.065,0.058,sleep
333,sleep_duration_s14:C(sex)[Female]:C(age_binary...,1.1529,0.000,0.065,0.058,sleep
356,sleep_end_s7:C(sex)[Female]:C(age_binary)[old],1.1564,0.000,0.056,0.051,sleep
...,...,...,...,...,...,...
634,travel_diameter_s14:C(sex)[Male]:C(age_binary)...,-0.9218,0.000,0.020,0.017,sociability
264,screen_unlocks_m14:C(sex)[Male]:C(age_binary)[...,-1.8067,0.000,0.020,0.018,device_usage
263,screen_unlocks_m14:C(sex)[Female]:C(age_binary...,-0.3754,0.020,0.020,0.018,device_usage
319,chronotype_score_s7:C(sex)[Male]:C(age_binary)...,-0.8183,0.007,0.020,0.018,sleep


In [32]:
feature_rankings_filter[feature_rankings_filter.r_squared > 0.02]['bucket'].value_counts()

bucket
sleep           35
activity        16
device_usage    15
sociability      5
Name: count, dtype: int64

In [33]:
aashi_df_working = feature_rankings_filter[(feature_rankings_filter.bucket == 'device_usage') | (feature_rankings_filter.bucket == 'sleep')]
aashi_df_working.shape

(54, 6)

In [34]:
aashi_df_working

,feature,coefficient,p_value,r_squared,adj_r_squared,bucket
353,sleep_end_s14:C(sex)[Female]:C(age_binary)[young],1.2404,0.000,0.068,0.061,sleep
351,sleep_end_s14:C(sex)[Female]:C(age_binary)[old],1.1913,0.001,0.068,0.061,sleep
331,sleep_duration_s14:C(sex)[Female]:C(age_binary...,0.9129,0.003,0.065,0.058,sleep
333,sleep_duration_s14:C(sex)[Female]:C(age_binary...,1.1529,0.000,0.065,0.058,sleep
356,sleep_end_s7:C(sex)[Female]:C(age_binary)[old],1.1564,0.000,0.056,0.051,sleep
358,sleep_end_s7:C(sex)[Female]:C(age_binary)[young],1.1271,0.000,0.056,0.051,sleep
401,sleep_start_m14:C(sex)[Female]:C(age_binary)[old],0.7311,0.010,0.051,0.044,sleep
403,sleep_start_m14:C(sex)[Female]:C(age_binary)[y...,0.7839,0.003,0.051,0.044,sleep
336,sleep_duration_s7:C(sex)[Female]:C(age_binary)...,0.9459,0.000,0.048,0.043,sleep
338,sleep_duration_s7:C(sex)[Female]:C(age_binary)...,0.9794,0.000,0.048,0.043,sleep


In [35]:
#save the df into csv file
aashi_df_working.to_csv('aashi_df_working.csv', index=False)

In [36]:
parag_df_working = feature_rankings_filter[(feature_rankings_filter.bucket == 'sociability') | (feature_rankings_filter.bucket == 'activity')]
parag_df_working.shape

(25, 6)

In [37]:
parag_df_working

,feature,coefficient,p_value,r_squared,adj_r_squared,bucket
481,time_at_home_m14:C(sex)[Female]:C(age_binary)[...,0.5099,0.001,0.029,0.027,sociability
484,time_at_home_m14:C(sex)[Male]:C(age_binary)[yo...,0.4858,0.022,0.029,0.027,sociability
483,time_at_home_m14:C(sex)[Female]:C(age_binary)[...,0.9207,0.000,0.029,0.027,sociability
194,walking_rate_s14:C(sex)[Male]:C(age_binary)[yo...,-1.0475,0.000,0.027,0.024,activity
192,walking_rate_s14:C(sex)[Male]:C(age_binary)[old],-0.9088,0.001,0.027,0.024,activity
488,time_at_home_m7:C(sex)[Female]:C(age_binary)[y...,0.8005,0.000,0.023,0.021,sociability
486,time_at_home_m7:C(sex)[Female]:C(age_binary)[old],0.4005,0.005,0.023,0.021,sociability
199,walking_rate_s7:C(sex)[Male]:C(age_binary)[young],-1.0692,0.000,0.022,0.020,activity
34,activity_percent_s14:C(sex)[Male]:C(age_binary...,-0.6893,0.001,0.022,0.019,activity
14,active_time_s14:C(sex)[Male]:C(age_binary)[young],-0.6893,0.001,0.022,0.019,activity


In [38]:
#save the df into csv file
parag_df_working.to_csv('parag_df_working.csv', index=False)

In [39]:
feature_rankings_filter[feature_rankings_filter['feature'].str.startswith('ste')]

,feature,coefficient,p_value,r_squared,adj_r_squared,bucket
142,step_count_m14:C(sex)[Male]:C(age_binary)[old],-1.2947,0.000,0.02,0.018,activity
143,step_count_m14:C(sex)[Female]:C(age_binary)[yo...,0.5407,0.002,0.02,0.018,activity


In [40]:
feature_rankings_filter[(feature_rankings_filter.bucket == 'device_usage') | (feature_rankings_filter.bucket == 'sleep')]

,feature,coefficient,p_value,r_squared,adj_r_squared,bucket
353,sleep_end_s14:C(sex)[Female]:C(age_binary)[young],1.2404,0.000,0.068,0.061,sleep
351,sleep_end_s14:C(sex)[Female]:C(age_binary)[old],1.1913,0.001,0.068,0.061,sleep
331,sleep_duration_s14:C(sex)[Female]:C(age_binary...,0.9129,0.003,0.065,0.058,sleep
333,sleep_duration_s14:C(sex)[Female]:C(age_binary...,1.1529,0.000,0.065,0.058,sleep
356,sleep_end_s7:C(sex)[Female]:C(age_binary)[old],1.1564,0.000,0.056,0.051,sleep
358,sleep_end_s7:C(sex)[Female]:C(age_binary)[young],1.1271,0.000,0.056,0.051,sleep
401,sleep_start_m14:C(sex)[Female]:C(age_binary)[old],0.7311,0.010,0.051,0.044,sleep
403,sleep_start_m14:C(sex)[Female]:C(age_binary)[y...,0.7839,0.003,0.051,0.044,sleep
336,sleep_duration_s7:C(sex)[Female]:C(age_binary)...,0.9459,0.000,0.048,0.043,sleep
338,sleep_duration_s7:C(sex)[Female]:C(age_binary)...,0.9794,0.000,0.048,0.043,sleep


In [41]:
df.sleep_duration_m7.isna().sum()

2133

In [42]:
750/2

375.0

In [48]:
feature_buckets

{'Sleep': ['sleep_duration_m7',
  'sleep_duration_m14',
  'sleep_duration_s7',
  'sleep_duration_s14',
  'sleep_start_m7',
  'sleep_start_m14',
  'sleep_start_s7',
  'sleep_start_s14',
  'sleep_end_m7',
  'sleep_end_m14',
  'sleep_end_s7',
  'sleep_end_s14',
  'sleep_interruptions_m7',
  'sleep_interruptions_m14',
  'sleep_interruptions_s7',
  'sleep_interruptions_s14',
  'sleep_routine_index_m7',
  'sleep_routine_index_m14',
  'sleep_routine_index_s7',
  'sleep_routine_index_s14',
  'chronotype_score_m7',
  'chronotype_score_m14',
  'chronotype_score_s7',
  'chronotype_score_s14'],
 'Activity': ['active_time_m7',
  'active_time_m14',
  'active_time_s7',
  'active_time_s14',
  'activity_percent_m7',
  'activity_percent_m14',
  'activity_percent_s7',
  'activity_percent_s14',
  'ec_Activity_m7',
  'ec_Activity_m14',
  'ec_Activity_s7',
  'ec_Activity_s14',
  'ec_PedometerData_m7',
  'ec_PedometerData_m14',
  'ec_PedometerData_s7',
  'ec_PedometerData_s14',
  'step_count_m7',
  'step_cou

In [86]:
feature_buckets['Sleep']

['sleep_duration_m7',
 'sleep_duration_m14',
 'sleep_duration_s7',
 'sleep_duration_s14',
 'sleep_start_m7',
 'sleep_start_m14',
 'sleep_start_s7',
 'sleep_start_s14',
 'sleep_end_m7',
 'sleep_end_m14',
 'sleep_end_s7',
 'sleep_end_s14',
 'sleep_interruptions_m7',
 'sleep_interruptions_m14',
 'sleep_interruptions_s7',
 'sleep_interruptions_s14',
 'sleep_routine_index_m7',
 'sleep_routine_index_m14',
 'sleep_routine_index_s7',
 'sleep_routine_index_s14',
 'chronotype_score_m7',
 'chronotype_score_m14',
 'chronotype_score_s7',
 'chronotype_score_s14']

In [64]:
results = {}

# Analyze missingness for each bucket
for bucket_name, cols in feature_buckets.items():
    # Filter columns that exist in the DataFrame
    valid_cols = [c for c in cols if c in df.columns]
    
    # Flag days with ANY missing data in this bucket
    df[f'missing_{bucket_name}'] = df[valid_cols].isnull().any(axis=1)
    
    # Compute missing proportion per user for this bucket
    missing_proportion = df.groupby('user_id')[f'missing_{bucket_name}'].mean()
    results[bucket_name] = missing_proportion

# Combine results into a DataFrame
missing_by_bucket = pd.DataFrame(results)
missing_by_bucket

,Sleep,Activity,Sociability,Device Usage,Demographic,Other,Target
user_id,,,,,,,
us-east-1:db841a21-c400-c09b-cb33-144516a80231,0.666667,0.333333,0.333333,0.166667,1.0,0.166667,0.0
us-east-1:db841a21-c400-ce85-5cd6-95d896fd3268,1.000000,1.000000,1.000000,1.000000,1.0,0.166667,0.0
us-east-1:db841a21-c402-c83f-fadf-41f6d4271053,1.000000,0.500000,0.500000,0.500000,1.0,0.500000,0.0
us-east-1:db841a21-c402-c9e0-501c-45fd38e3b5e3,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,0.0
us-east-1:db841a21-c402-cf75-e0bb-eec187eb84b1,1.000000,0.500000,0.750000,1.000000,1.0,0.250000,0.0
...,...,...,...,...,...,...,...
us-east-1:db841a21-c4ff-c644-6a64-0583e73a79d9,1.000000,0.800000,0.800000,0.800000,1.0,0.200000,0.0
us-east-1:db841a21-c4ff-c868-02a6-5cfff64a075a,1.000000,0.333333,0.500000,0.500000,1.0,0.166667,0.0
us-east-1:db841a21-c4ff-cb45-cad0-4821d631eb58,0.833333,0.333333,0.500000,0.166667,1.0,0.166667,0.0


In [101]:
a = df[df.user_id == 'us-east-1:db841a21-c400-c09b-cb33-144516a80231'][['user_id', 'ts', 'sleep_duration_m7', 'sleep_duration_m14', 'sleep_duration_s7', 'sleep_duration_s14', 'sleep_start_m7', 'sleep_start_m14', 'sleep_start_s7', 'sleep_start_s14', 'sleep_end_m7', 'sleep_end_m14', 'sleep_end_s7', 'sleep_end_s14', 'sleep_interruptions_m7', 'sleep_interruptions_m14', 'sleep_interruptions_s7', 'sleep_interruptions_s14', 'sleep_routine_index_m7', 'sleep_routine_index_m14', 'sleep_routine_index_s7', 'sleep_routine_index_s14', 'chronotype_score_m7', 'chronotype_score_m14', 'chronotype_score_s7', 'chronotype_score_s14']]
a

,user_id,ts,sleep_duration_m7,sleep_duration_m14,sleep_duration_s7,sleep_duration_s14,sleep_start_m7,sleep_start_m14,sleep_start_s7,sleep_start_s14,...,sleep_interruptions_s7,sleep_interruptions_s14,sleep_routine_index_m7,sleep_routine_index_m14,sleep_routine_index_s7,sleep_routine_index_s14,chronotype_score_m7,chronotype_score_m14,chronotype_score_s7,chronotype_score_s14
456,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-05-04 14:13:46.887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
893,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-05-18 14:39:27.530,32057.142857,NaN,5605.609435,NaN,21000.000000,NaN,3766.961640,NaN,...,0.0,NaN,0.928241,NaN,0.020512,NaN,1.425714,NaN,0.047909,NaN
1322,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-06-01 14:39:35.855,30042.857143,30042.857143,4296.842695,6041.231954,22285.714286,21964.285714,2953.206490,5573.095693,...,0.0,0.0,0.907655,0.914641,0.024392,0.025047,1.592857,1.547143,0.013801,0.090590
1847,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-06-17 23:17:52.098,32528.571429,NaN,3722.326461,NaN,20957.142857,NaN,2629.231284,NaN,...,0.0,NaN,0.916700,0.925017,0.010720,0.013616,1.540000,1.539286,0.014142,0.029733
2455,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-07-09 12:28:52.414,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.935317,0.932482,0.007820,0.009945,1.531429,1.562143,0.047409,0.045434
2594,us-east-1:db841a21-c400-c09b-cb33-144516a80231,2024-07-16 14:19:11.857,32400.000000,33428.571429,2754.995463,3643.419478,20528.571429,19928.571429,2141.428095,2013.198210,...,0.0,0.0,0.942874,0.937847,0.013081,0.012496,1.464286,1.490714,0.022254,0.040281


In [102]:
for i in a.index:
    print(a.loc[i])

user_id                    us-east-1:db841a21-c400-c09b-cb33-144516a80231
ts                                             2024-05-04 14:13:46.887000
sleep_duration_m7                                                     NaN
sleep_duration_m14                                                    NaN
sleep_duration_s7                                                     NaN
sleep_duration_s14                                                    NaN
sleep_start_m7                                                        NaN
sleep_start_m14                                                       NaN
sleep_start_s7                                                        NaN
sleep_start_s14                                                       NaN
sleep_end_m7                                                          NaN
sleep_end_m14                                                         NaN
sleep_end_s7                                                          NaN
sleep_end_s14                         

In [73]:
# Example: Find users with >80% missing days in the 'sleep' bucket
threshold = 0.3
problematic_users_sleep = missing_by_bucket[missing_by_bucket['Sleep'] > threshold].index.tolist()
problematic_users_sleep

['us-east-1:db841a21-c400-c09b-cb33-144516a80231',
 'us-east-1:db841a21-c400-ce85-5cd6-95d896fd3268',
 'us-east-1:db841a21-c402-c83f-fadf-41f6d4271053',
 'us-east-1:db841a21-c402-c9e0-501c-45fd38e3b5e3',
 'us-east-1:db841a21-c402-cf75-e0bb-eec187eb84b1',
 'us-east-1:db841a21-c403-c5dc-0f8d-fdaad2b0c8c0',
 'us-east-1:db841a21-c403-ce6b-e911-a2865b0f3c9a',
 'us-east-1:db841a21-c404-c304-c2e4-572d6e178c68',
 'us-east-1:db841a21-c404-c440-4199-c74ef9e1b19d',
 'us-east-1:db841a21-c404-c761-77c2-8cab76aaa00a',
 'us-east-1:db841a21-c404-c936-f3ee-306b897c08c3',
 'us-east-1:db841a21-c404-cb9a-95f4-a148924f855c',
 'us-east-1:db841a21-c404-cff1-369e-124fb7a439b3',
 'us-east-1:db841a21-c405-c5e7-6ec9-d6661f205f9d',
 'us-east-1:db841a21-c405-ccc6-81c9-f5a0ba288ba5',
 'us-east-1:db841a21-c406-c96a-f0ab-fc0e42567bc7',
 'us-east-1:db841a21-c406-ccf9-7dce-71496e2b0b9c',
 'us-east-1:db841a21-c407-c0e5-a83d-7cd2cd6853b7',
 'us-east-1:db841a21-c408-c249-6b08-c8edb187f610',
 'us-east-1:db841a21-c408-c62c-

In [74]:
len(problematic_users_sleep)

591

In [78]:
df.head()

,user_id,ts,active_time_m7,active_time_m14,active_time_s7,active_time_s14,activity_percent_m7,activity_percent_m14,activity_percent_s7,activity_percent_s14,...,phq9_total,deployment,age_binary,missing_Sleep,missing_Activity,missing_Sociability,missing_Device Usage,missing_Demographic,missing_Other,missing_Target
0,us-east-1:db841a21-c4fd-cf09-4379-5af0b6185e0e,2024-03-23 21:58:52.242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24.0,hr-ascent-1,old,True,True,True,True,True,True,False
1,us-east-1:db841a21-c48b-cd1c-563f-eb23aa6c2768,2024-03-26 14:46:43.386,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,hr-ascent-1,young,True,True,True,True,True,True,False
2,us-east-1:db841a21-c46a-c6fd-ca13-1faabac2c46d,2024-03-27 16:19:40.526,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.0,hr-ascent-1,old,True,True,True,True,True,True,False
3,us-east-1:db841a21-c4b2-c6b9-91ed-370b2da2aa86,2024-03-27 16:42:15.485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.0,hr-ascent-1,young,True,True,True,True,True,True,False
4,us-east-1:db841a21-c438-c76a-5078-c47a75e6b0c3,2024-03-28 14:52:36.017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,hr-ascent-1,young,True,True,True,True,True,True,False


In [105]:
feature_buckets.keys()

dict_keys(['Sleep', 'Activity', 'Sociability', 'Device Usage', 'Demographic', 'Other', 'Target'])

In [106]:
critical_features = feature_buckets['Sleep'] + feature_buckets['Activity'] + feature_buckets['Sociability'] 
critical_features

['sleep_duration_m7',
 'sleep_duration_m14',
 'sleep_duration_s7',
 'sleep_duration_s14',
 'sleep_start_m7',
 'sleep_start_m14',
 'sleep_start_s7',
 'sleep_start_s14',
 'sleep_end_m7',
 'sleep_end_m14',
 'sleep_end_s7',
 'sleep_end_s14',
 'sleep_interruptions_m7',
 'sleep_interruptions_m14',
 'sleep_interruptions_s7',
 'sleep_interruptions_s14',
 'sleep_routine_index_m7',
 'sleep_routine_index_m14',
 'sleep_routine_index_s7',
 'sleep_routine_index_s14',
 'chronotype_score_m7',
 'chronotype_score_m14',
 'chronotype_score_s7',
 'chronotype_score_s14',
 'active_time_m7',
 'active_time_m14',
 'active_time_s7',
 'active_time_s14',
 'activity_percent_m7',
 'activity_percent_m14',
 'activity_percent_s7',
 'activity_percent_s14',
 'ec_Activity_m7',
 'ec_Activity_m14',
 'ec_Activity_s7',
 'ec_Activity_s14',
 'ec_PedometerData_m7',
 'ec_PedometerData_m14',
 'ec_PedometerData_s7',
 'ec_PedometerData_s14',
 'step_count_m7',
 'step_count_m14',
 'step_count_s7',
 'step_count_s14',
 'steps_most_activ

In [107]:
# Select critical features (e.g., all sleep-related columns)
#critical_features = feature_buckets['sleep'] + feature_buckets['travel'] + [...]  # Customize

# Compute missingness for each feature
missing_by_feature = df.groupby('user_id')[critical_features].apply(
    lambda x: x.isnull().mean()
).reset_index()
missing_by_feature

,user_id,sleep_duration_m7,sleep_duration_m14,sleep_duration_s7,sleep_duration_s14,sleep_start_m7,sleep_start_m14,sleep_start_s7,sleep_start_s14,sleep_end_m7,...,travel_diameter_s7,travel_diameter_s14,ec_Location_m7,ec_Location_m14,ec_Location_s7,ec_Location_s14,total_location_duration_m7,total_location_duration_m14,total_location_duration_s7,total_location_duration_s14
0,us-east-1:db841a21-c400-c09b-cb33-144516a80231,0.333333,0.666667,0.333333,0.666667,0.333333,0.666667,0.333333,0.666667,0.333333,...,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
1,us-east-1:db841a21-c400-ce85-5cd6-95d896fd3268,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.000000,0.166667,0.000000,0.166667,0.000000,0.166667,0.000000,0.166667
2,us-east-1:db841a21-c402-c83f-fadf-41f6d4271053,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000
3,us-east-1:db841a21-c402-c9e0-501c-45fd38e3b5e3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,us-east-1:db841a21-c402-cf75-e0bb-eec187eb84b1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.500000,0.750000,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
596,us-east-1:db841a21-c4ff-c644-6a64-0583e73a79d9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.800000,0.800000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
597,us-east-1:db841a21-c4ff-c868-02a6-5cfff64a075a,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.333333,0.333333,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
598,us-east-1:db841a21-c4ff-cb45-cad0-4821d631eb58,0.666667,0.833333,0.666667,0.833333,0.666667,0.833333,0.666667,0.833333,0.666667,...,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
599,us-east-1:db841a21-c4ff-cb63-a6da-f45b3c81a5c5,0.333333,0.500000,0.333333,0.500000,0.333333,0.500000,0.333333,0.500000,0.333333,...,0.333333,0.500000,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667


In [109]:

# Melt for visualization/analysis
missing_by_feature_melted = missing_by_feature.melt(
    id_vars='user_id', var_name='feature', value_name='missing_proportion'
)
missing_by_feature_melted

,user_id,feature,missing_proportion
0,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_duration_m7,0.333333
1,us-east-1:db841a21-c400-ce85-5cd6-95d896fd3268,sleep_duration_m7,1.000000
2,us-east-1:db841a21-c402-c83f-fadf-41f6d4271053,sleep_duration_m7,1.000000
3,us-east-1:db841a21-c402-c9e0-501c-45fd38e3b5e3,sleep_duration_m7,1.000000
4,us-east-1:db841a21-c402-cf75-e0bb-eec187eb84b1,sleep_duration_m7,1.000000
...,...,...,...
64903,us-east-1:db841a21-c4ff-c644-6a64-0583e73a79d9,total_location_duration_s14,0.200000
64904,us-east-1:db841a21-c4ff-c868-02a6-5cfff64a075a,total_location_duration_s14,0.166667
64905,us-east-1:db841a21-c4ff-cb45-cad0-4821d631eb58,total_location_duration_s14,0.166667
64906,us-east-1:db841a21-c4ff-cb63-a6da-f45b3c81a5c5,total_location_duration_s14,0.166667


In [110]:
missing_by_feature_melted[missing_by_feature_melted.user_id == 'us-east-1:db841a21-c400-c09b-cb33-144516a80231']

,user_id,feature,missing_proportion
0,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_duration_m7,0.333333
601,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_duration_m14,0.666667
1202,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_duration_s7,0.333333
1803,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_duration_s14,0.666667
2404,us-east-1:db841a21-c400-c09b-cb33-144516a80231,sleep_start_m7,0.333333
...,...,...,...
61903,us-east-1:db841a21-c400-c09b-cb33-144516a80231,ec_Location_s14,0.166667
62504,us-east-1:db841a21-c400-c09b-cb33-144516a80231,total_location_duration_m7,0.166667
63105,us-east-1:db841a21-c400-c09b-cb33-144516a80231,total_location_duration_m14,0.166667
63706,us-east-1:db841a21-c400-c09b-cb33-144516a80231,total_location_duration_s7,0.166667
